In [42]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rioxr

from sklearn.metrics import confusion_matrix

In [40]:
pts = gpd.read_file(os.path.join('data','2025-sample-points','validation_sample_points_2025.shp'))
pts

,class,raster,LON,LAT,eva,geometry
0,3,2025_rawmodel.tif,-120.457684,34.449840,3,POINT (-120.45768 34.44984)
1,1,2025_rawmodel.tif,-120.472756,34.472097,1,POINT (-120.47276 34.47210)
2,1,2025_rawmodel.tif,-120.464888,34.465340,1,POINT (-120.46489 34.46534)
3,3,2025_rawmodel.tif,-120.454276,34.460803,3,POINT (-120.45428 34.46080)
4,1,2025_rawmodel.tif,-120.466544,34.459381,1,POINT (-120.46654 34.45938)
...,...,...,...,...,...,...
707,1,2025_rawmodel.tif,-120.468722,34.464030,1,POINT (-120.46872 34.46403)
708,3,2025_rawmodel.tif,-120.452600,34.459870,3,POINT (-120.45260 34.45987)
709,1,2025_rawmodel.tif,-120.467996,34.462554,1,POINT (-120.46800 34.46255)
710,1,2025_rawmodel.tif,-120.464310,34.461564,1,POINT (-120.46431 34.46156)


In [50]:
map = rioxr.open_rasterio(os.path.join('data','2025_rawmodel_matched.tif')).drop_vars('band')
map

<xarray.DataArray (band: 1, y: 5931, x: 5071)> Size: 60MB
[30076101 values with dtype=int16]
Coordinates:
  * x            (x) float64 41kB 7.318e+05 7.318e+05 ... 7.348e+05 7.348e+05
  * y            (y) float64 47kB 3.818e+06 3.818e+06 ... 3.814e+06 3.814e+06
    spatial_ref  int64 8B 0
Dimensions without coordinates: band
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     0
    scale_factor:   1.0
    add_offset:     0.0

In [93]:
# Labels extracted from raster via QGIS
# 0 = unclassified (unclear if that's the case)
# 1 = iceplant
# 2 = other
# 3 = other vegetation

counts = pd.DataFrame(*np.unique(map, return_counts = True))
counts = counts.reset_index(names='n').drop(columns=0)
counts.index = counts.index.rename('class')
counts

,n
class,
0,378461
1,6243501
2,8545213
3,14908926


In [99]:
total_pixels = counts['n'].sum()
# Convert to from meters squared to hectares, 1 pixel = 0.6x0.6m (from QGIS)
total_area = total_pixels * (0.6)**2 * 0.0001
counts['perc coverage'] = counts['n']/total_pixels*100
counts['hectares'] = total_area * counts['perc coverage']/100
counts

,n,fraction,perc coverage,hectares
class,,,,
0,378461,0.012583,1.258345,13.624596
1,6243501,0.207590,20.759011,224.766036
2,8545213,0.284120,28.411971,307.627668
3,14908926,0.495707,49.570674,536.721336


In [94]:
pd.DataFrame({'map': pts['class'].value_counts(),
              'reviewed': pts['eva'].value_counts()})

,map,reviewed
1,300,334
2,112,115
3,300,263


In [37]:
# Notation in sklearn is transposed from Olofsson
n = confusion_matrix(pts['eva'], pts['class']).T
n


array([[292,   2,   6],
       [  1, 111,   0],
       [ 41,   2, 257]])

In [ ]:
# Percentage of area estimation

